# Python 기초 3 · 직접 풀어보는 연습문제 정답지

이 노트북은 `Python_Basic_3_Functions_Modules_Practice.ipynb`의 모든 연습문제에 대한 **표준 모범 코드와 결과 해석**을 제공합니다.


In [ ]:
# [실습 환경 준비]
# 이 셀은 실습을 위한 사전 데이터를 정의합니다. 셀을 먼저 실행해주세요.
mock_users = [
    {"id": "U01", "name": "Alice", "clicks": 120, "purchases": 6, "review_score": 4.2},
    {"id": "U02", "name": "Bob", "clicks": 45, "purchases": 0, "review_score": 3.8},
    {"id": "U03", "name": "Charlie", "clicks": 300, "purchases": 25, "review_score": 4.9},
    {"id": "U04", "name": "David", "clicks": 0, "purchases": 0, "review_score": 2.5},
    {"id": "U05", "name": "Emma", "clicks": 210, "purchases": 14, "review_score": 4.7}
]

dirty_tags = [" #DataScience  ", "PYTHON_3 ", "  Machine-learning! ", " PANDAS "]


## 문제 1. 기초 통계 요약 함수 작성 (평균, 중앙값, 분산) [함수 설계]

숫자 리스트를 입력받아 다음 통계량을 딕셔너리 형태로 반환하는 함수 `calc_descriptive_stats(data)`를 작성하세요.
- `mean`: 산술 평균
- `median`: 중앙값 (데이터를 정렬한 후 홀수 개면 정중앙 값, 짝수 개면 중앙 2개 값의 평균)
- `variance`: 표본 분산 (각 편차 제곱의 합 / (N - 1), 단 N > 1인 경우)

테스트 데이터: `sample_data = [12, 15, 18, 20, 22, 25, 28, 30]`


In [ ]:
# 문제 1 풀이
def calc_descriptive_stats(data):
    if not data:
        return {"mean": 0, "median": 0, "variance": 0}
    
    n = len(data)
    # 1. 평균
    mean_val = sum(data) / n
    
    # 2. 중앙값
    sorted_d = sorted(data)
    if n % 2 != 0:
        median_val = sorted_d[n // 2]
    else:
        median_val = (sorted_d[n // 2 - 1] + sorted_d[n // 2]) / 2.0
        
    # 3. 표본 분산
    if n > 1:
        variance_val = sum((x - mean_val) ** 2 for x in data) / (n - 1)
    else:
        variance_val = 0.0
        
    return {
        "count": n,
        "mean": round(mean_val, 2),
        "median": round(median_val, 2),
        "variance": round(variance_val, 2)
    }

sample_data = [12, 15, 18, 20, 22, 25, 28, 30]
result = calc_descriptive_stats(sample_data)
print("요약 통계 결과:", result)


**결과 해석 / 확인한 점:**  
1. **짝수 개 데이터의 중앙값 산출:** 길이가 8인 리스트에서 중앙값은 4번째(20)와 5번째(22)의 산술 평균인 21.0이 정확히 계산되었습니다.
2. **표본 분산(Bessel's Correction):** 모집단 분산(N)과 달리 표본 분산은 자유도 `n - 1`로 나누어 모분산의 불편추정량(Unbiased Estimator)을 도출했습니다.


## 문제 2. 전환율(Conversion Rate) 계산 및 기본값 인자 [비즈니스 함수]

웹사이트 전환율(CR)을 계산하는 함수 `calc_conversion_rate(conversions, visitors, as_percentage=True, decimals=2)`를 작성하세요.
- 요구사항:
  1. `visitors`가 0 이하일 경우 0 나눗셈 에러(`ZeroDivisionError`)를 방지하고 `0.0`을 반환해야 합니다.
  2. `as_percentage=True`이면 100을 곱해 퍼센트(%) 단위로 반환하고, `False`이면 0 ~ 1 사이의 비율로 반환합니다.
  3. `decimals` 자리수만큼 반올림합니다.
- `setup`의 `mock_users` 리스트에서 각 사용자의 전환율(`purchases / clicks`)을 위 함수로 계산하여 출력하세요.


In [ ]:
# 문제 2 풀이
def calc_conversion_rate(conversions, visitors, as_percentage=True, decimals=2):
    if visitors <= 0:
        return 0.0
    rate = conversions / visitors
    if as_percentage:
        rate *= 100
    return round(rate, decimals)

print("=== 고객별 구매 전환율(CR) 집계 ===")
for u in mock_users:
    cr = calc_conversion_rate(u["purchases"], u["clicks"])
    print(f"고객 {u['name']} (ID: {u['id']}): 클릭 {u['clicks']}회, 구매 {u['purchases']}건 -> CR: {cr}%")


**결과 해석 / 확인한 점:**  
1. **0 나눗셈 방어 로직:** ID U04(David)의 경우 클릭 수가 0인데도 프로그램이 중단되지 않고 0.0%로 안전하게 처리되었습니다.
2. **지표 비교:** Charlie의 경우 클릭 300회 중 25회 구매로 8.33%의 우수한 전환율을 기록했습니다.


## 문제 3. 람다(Lambda)와 sorted()를 이용한 다중 기준 정렬 [데이터 정렬]

`mock_users` 고객 목록을 다음 우선순위에 따라 정렬하여 출력하세요:
- **1순위:** `purchases`(구매 건수) 내림차순 (구매가 많은 고객 우선)
- **2순위:** 구매 건수가 같을 경우 `review_score`(평점) 내림차순

힌트: 튜플을 반환하는 람다 함수 `key=lambda x: (-x['purchases'], -x['review_score'])`를 활용할 수 있습니다.


In [ ]:
# 문제 3 풀이
# 람다식 다중 정렬: 마이너스 부호(-)를 붙여 내림차순 구현
ranked_users = sorted(mock_users, key=lambda x: (-x["purchases"], -x["review_score"]))

print("=== 구매 건수 및 평점 기준 랭킹 ===")
for rank, u in enumerate(ranked_users, start=1):
    print(f"{rank}등: {u['name']} | 구매 {u['purchases']}건 | 평점 {u['review_score']}점")


**결과 해석 / 확인한 점:**  
1. **튜플 키를 활용한 다중 정렬:** 파이썬은 튜플의 첫 번째 요소를 먼저 비교하고, 동률일 경우 두 번째 요소를 비교하는 특성이 있습니다. 수치형 데이터에 음수 부호(`-`)를 취하면 손쉽게 내림차순 다중 정렬이 가능합니다.
2. **순위 도출:** Charlie(25건) -> Emma(14건) -> Alice(6건) -> Bob(0건, 3.8점) -> David(0건, 2.5점) 순으로 완벽하게 서열화되었습니다.


## 문제 4. filter()와 lambda를 이용한 고가치 유저 추출 [함수형 필터링]

`mock_users`에서 다음 두 가지 조건을 **모두 만족**하는 '고가치 충성 고객(VIP)'만 `filter()`와 `lambda`를 써서 추출하세요:
1. `purchases >= 10` (구매 10건 이상)
2. `review_score >= 4.5` (리뷰 평점 4.5 이상)

추출된 고객들의 이름과 주요 지표를 리스트 형태로 출력하세요.


In [ ]:
# 문제 4 풀이
vip_filter = filter(lambda u: u["purchases"] >= 10 and u["review_score"] >= 4.5, mock_users)
vip_list = list(vip_filter)

print("=== 고가치 충성 고객(VIP) 추출 결과 ===")
for v in vip_list:
    print(f"ID: {v['id']}, 이름: {v['name']}, 구매: {v['purchases']}건, 평점: {v['review_score']}")


**결과 해석 / 확인한 점:**  
1. **filter() 함수의 이점:** 반복문 없이 데이터 파이프라인 스타일로 참인 요소만 이터레이터로 반환하여 메모리 효율적인 데이터 서브셋 추출이 가능합니다.
2. **세그먼트 검증:** 구매 25건에 4.9점인 Charlie와 구매 14건에 4.7점인 Emma 2명의 핵심 타깃 고객이 정확히 분리되었습니다.


## 문제 5. *args를 활용한 유연한 지표 가중합 함수 [가변 인자]

임의의 개수의 지표 점수들을 전달받아 산술 평균을 계산하는 `calc_average_metric(*scores)` 함수를 작성하세요.
- 요구사항:
  1. 점수가 하나도 전달되지 않은 경우 `0.0`을 반환합니다.
  2. 음수 점수가 포함되어 있으면 무시하고 0 이상의 유효 점수만으로 평균을 냅니다.

테스트 케이스:
- `calc_average_metric(80, 90, 100)` -> 90.0
- `calc_average_metric(50, -10, 70, -99, 90)` -> 70.0 (50, 70, 90의 평균)
- `calc_average_metric()` -> 0.0


In [ ]:
# 문제 5 풀이
def calc_average_metric(*scores):
    valid_scores = [s for s in scores if s >= 0]
    if not valid_scores:
        return 0.0
    return round(sum(valid_scores) / len(valid_scores), 2)

print("테스트 1:", calc_average_metric(80, 90, 100))
print("테스트 2:", calc_average_metric(50, -10, 70, -99, 90))
print("테스트 3:", calc_average_metric())


**결과 해석 / 확인한 점:**  
1. ***args 가변 인자의 유연성:** 인자의 개수를 사전에 확정하기 어려운 분석 유틸리티 함수에서 파라미터를 튜플로 패킹하여 유연하게 처리할 수 있습니다.
2. **예외 방어:** 음수 이상치를 필터링하고 인자가 전달되지 않은 엣지 케이스에서도 0 나눗셈 없이 안전하게 동작함을 확인했습니다.


## 문제 6. 텍스트 정제 전처리 함수 모듈화 [텍스트 ETL]

자연어 처리(NLP)나 키워드 분석 전단계로 태그 문자열을 정제하는 함수 `clean_tag(tag)`를 작성하세요.
- 요구사항:
  1. 앞뒤 공백을 제거(`strip()`)합니다.
  2. `#`, `!`, `-` 등 불필요한 특수문자를 제거합니다.
  3. 모든 문자를 소문자로 변환(`lower()`)합니다.
- `setup` 코드의 `dirty_tags = [" #DataScience  ", "PYTHON_3 ", "  Machine-learning! ", " PANDAS "]`에 `map()` 함수로 `clean_tag`를 적용한 정제 리스트를 출력하세요.


In [ ]:
# 문제 6 풀이
def clean_tag(tag):
    cleaned = tag.strip().lower()
    for char in ["#", "!", "-"]:
        cleaned = cleaned.replace(char, "")
    return cleaned.strip()

cleaned_tag_list = list(map(clean_tag, dirty_tags))
print("정제 전:", dirty_tags)
print("정제 후:", cleaned_tag_list)


**결과 해석 / 확인한 점:**  
1. **단위 기능의 모듈화:** 전처리 규칙을 단일 함수 `clean_tag()`로 캡슐화해 두면 나중에 다른 데이터셋이나 대용량 판다스 시리즈(`.apply(clean_tag)`)에 그대로 재사용할 수 있습니다.
2. **데이터 일관성:** 불필요한 노이즈 문자와 대소문자 불일치가 제거되어 `"datascience"`, `"python_3"`, `"machinelearning"`, `"pandas"`로 완벽히 정규화되었습니다.
